**Utiliza los conceptos aprendidos en los laboratorios de regresión y clasificación para encontrar el error estándar de los coeficientes de una regresión (lineal/logística) simple para los datasets de “Advertising” y “Default”.**

In [132]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
import statsmodels.formula.api as smf
from sklearn.linear_model import Ridge
from skopt import BayesSearchCV
from sklearn.model_selection import KFold

# Advertising

In [18]:
df = pd.read_csv("Advertising.csv", index_col="Unnamed: 0")

In [20]:
df.head()

,TV,radio,newspaper,sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9


In [52]:
X = df[["TV", "radio", "newspaper"]]
X = sm.add_constant(X)

y = df["sales"]

modelo = sm.OLS(y, X).fit()

errores_estandar = dict(zip(modelo.params.index, modelo.bse))

print(errores_estandar)


# Resultados
# print(modelo.summary())


{'const': 0.3119082363217911, 'TV': 0.0013948968069749745, 'radio': 0.008611233967301946, 'newspaper': 0.005871009647086371}


# Defalut

In [118]:
df2 = pd.read_csv("Default.csv")


df2["default"] = df2["default"].map({"No": 0, "Yes": 1})
df2["student"] = df2["student"].map({"No": 0, "Yes": 1})

In [120]:
df2.head()

,default,student,balance,income
0,0,0,729.526495,44361.625074
1,0,1,817.180407,12106.134700
2,0,0,1073.549164,31767.138950
3,0,0,529.250605,35704.493940
4,0,0,785.655883,38463.495880


In [122]:
modelo = smf.logit("default ~ C(student) + balance + income", data=df2).fit(disp=False)

# Extraer coeficientes y errores estándar
coef = modelo.params
stderr = modelo.bse

# Mapear coeficiente con error estándar
mapeo = dict(zip(coef.index, stderr))

print("\nMapeo coeficiente error estándar:")
print(mapeo)

# Mapear coeficientes (betas)
coeficientes = dict(zip(modelo.params.index, modelo.params))

print("\nCoeficientes del modelo (betas):")
print(coeficientes)

# Mostrar resultados
# print(modelo.summary())


Mapeo coeficiente error estándar:
{'Intercept': 0.49227264972939516, 'C(student)[T.1]': 0.23625692636743892, 'balance': 0.00023190442570016984, 'income': 8.202765618607335e-06}

Coeficientes del modelo (betas):
{'Intercept': -10.869045212324698, 'C(student)[T.1]': -0.6467758082222663, 'balance': 0.005736505265483497, 'income': 3.0334501191634992e-06}


**Utiliza bootstrap para simular 1000 remuestreos de esos datasets y calcula la media de los coeficientes obtenidos al aplicarle regresión a cada remuestreo. Calcula la desviación estándar.**

In [ ]:
from sklearn.utils import resample

# Advertising

In [111]:
df = pd.read_csv("Advertising.csv", index_col="Unnamed: 0")

# Definir variables
X = df[["TV", "radio", "newspaper"]]
X = sm.add_constant(X)  # agregar intercepto
y = df["sales"]

# Ajustar modelo original
modelo = sm.OLS(y, X).fit()

# Errores estándar del modelo original
errores_estandar = dict(zip(modelo.params.index, modelo.bse))
print("Errores estándar originales:")
print(errores_estandar)


# Coeficientes del modelo
coeficientes = dict(zip(modelo.params.index, modelo.params))
print("Coeficientes del modelo (betas):")
print(coeficientes)


Errores estándar originales:
{'const': 0.3119082363217911, 'TV': 0.0013948968069749745, 'radio': 0.008611233967301946, 'newspaper': 0.005871009647086371}
Coeficientes del modelo (betas):
{'const': 2.9388893694594134, 'TV': 0.04576464545539757, 'radio': 0.18853001691820445, 'newspaper': -0.001037493042476259}


In [107]:
# Bootstrap
n_boot = 1000
coef_bootstrap = []

for i in range(n_boot):
    # Remuestreo con reemplazo
    df_sample = resample(df, replace=True, n_samples=len(df), random_state=i)
    
    # Variables para este remuestreo
    X_sample = df_sample[["TV", "radio", "newspaper"]]
    X_sample = sm.add_constant(X_sample)
    y_sample = df_sample["sales"]
    
    # Ajustar modelo OLS
    modelo_boot = sm.OLS(y_sample, X_sample).fit()
    
    # Guardar coeficientes
    coef_bootstrap.append(modelo_boot.params.values)


In [108]:
# Convertir a array numpy
coef_bootstrap = np.array(coef_bootstrap)

# Calcular media y desviación estándar
media_coef = np.mean(coef_bootstrap, axis=0)
std_coef = np.std(coef_bootstrap, axis=0, ddof=1)

# Mapear con nombres de coeficientes
nombres_coef = modelo.params.index
media_coef_dict = dict(zip(nombres_coef, media_coef))
std_coef_dict = dict(zip(nombres_coef, std_coef))

print("\nMedia de los coeficientes (bootstrap):")
print(media_coef_dict)

print("\nDesviación estándar de los coeficientes (bootstrap):")
print(std_coef_dict)


Media de los coeficientes (bootstrap):
{'const': 2.952918112272016, 'TV': 0.04565970689833001, 'radio': 0.18814721517677105, 'newspaper': -0.0005439756675229768}

Desviación estándar de los coeficientes (bootstrap):
{'const': 0.33803906752727747, 'TV': 0.0019077417970319883, 'radio': 0.010955006809129579, 'newspaper': 0.006294971133078852}


# Default

In [97]:
df2 = pd.read_csv("Default.csv")


df2["default"] = df2["default"].map({"No": 0, "Yes": 1})
df2["student"] = df2["student"].map({"No": 0, "Yes": 1})

In [99]:
# Número de remuestreos bootstrap
n_boot = 1000

# Guardar coeficientes de cada bootstrap
coef_bootstrap = []

for i in range(n_boot):
    # Crear remuestreo con reemplazo
    df_sample = resample(df2, replace=True, n_samples=len(df2), random_state=i)
    
    # Ajustar regresión logística
    modelo = smf.logit("default ~ C(student) + balance + income", data=df_sample).fit(disp=False)
    
    # Guardar coeficientes
    coef_bootstrap.append(modelo.params.values)

In [103]:
# Convertir a array numpy para facilidad de cálculo
coef_bootstrap = np.array(coef_bootstrap)

# Calcular media y desviación estándar de los coeficientes
media_coef = np.mean(coef_bootstrap, axis=0)
std_coef = np.std(coef_bootstrap, axis=0, ddof=1)  # ddof=1 para estimación insesgada

# Asociar nombres de coeficientes
nombres_coef = modelo.params.index
media_coef_dict = dict(zip(nombres_coef, media_coef))
std_coef_dict = dict(zip(nombres_coef, std_coef))

print("Media de los coeficientes (bootstrap):")
print(media_coef_dict)

print("\nDesviación estándar de los coeficientes (bootstrap):")
print(std_coef_dict)

Media de los coeficientes (bootstrap):
{'Intercept': -10.931311490182797, 'C(student)[T.1]': -0.6370725667406434, 'balance': 0.005762413395217705, 'income': 3.488286258300028e-06}

Desviación estándar de los coeficientes (bootstrap):
{'Intercept': 0.5093489815120217, 'C(student)[T.1]': 0.2451181793852033, 'balance': 0.00023610987574669955, 'income': 8.383030443086629e-06}


**Compara los resultados obtenidos con el método visto en los laboratorios contra los resultados obtenidos con bootstrap. ¿Por qué podría haber diferencias en los resultados?**

# Advertising

In [126]:
# Datos advertising
coef_originales = {'const': 2.9388893694594134, 
                   'TV': 0.04576464545539757, 
                   'radio': 0.18853001691820445, 
                   'newspaper': -0.001037493042476259}

errores_estandar_originales = {'const': 0.3119082363217911, 
                               'TV': 0.0013948968069749745, 
                               'radio': 0.008611233967301946, 
                               'newspaper': 0.005871009647086371}

media_bootstrap = {'const': 2.952918112272016, 
                   'TV': 0.04565970689833001, 
                   'radio': 0.18814721517677105, 
                   'newspaper': -0.0005439756675229768}

std_bootstrap = {'const': 0.33803906752727747, 
                 'TV': 0.0019077417970319883, 
                 'radio': 0.010955006809129579, 
                 'newspaper': 0.006294971133078852}

# Crear DataFrame
df_comparacion_advertising = pd.DataFrame({
    'Coeficiente original': coef_originales,
    'Error estándar original': errores_estandar_originales,
    'Media bootstrap': media_bootstrap,
    'Desviación estándar bootstrap': std_bootstrap
})

df_comparacion_advertising


,Coeficiente original,Error estándar original,Media bootstrap,Desviación estándar bootstrap
const,2.938889,0.311908,2.952918,0.338039
TV,0.045765,0.001395,0.045660,0.001908
radio,0.188530,0.008611,0.188147,0.010955
newspaper,-0.001037,0.005871,-0.000544,0.006295


# Default

In [124]:
# Datos default
errores_estandar_originales = {
    'Intercept': 0.49227264972939516,
    'C(student)[T.1]': 0.23625692636743892,
    'balance': 0.00023190442570016984,
    'income': 8.202765618607335e-06
}

coef_originales = {
    'Intercept': -10.869045212324698,
    'C(student)[T.1]': -0.6467758082222663,
    'balance': 0.005736505265483497,
    'income': 3.0334501191634992e-06
}

media_bootstrap = {
    'Intercept': -10.931311490182797,
    'C(student)[T.1]': -0.6370725667406434,
    'balance': 0.005762413395217705,
    'income': 3.488286258300028e-06
}

std_bootstrap = {
    'Intercept': 0.5093489815120217,
    'C(student)[T.1]': 0.2451181793852033,
    'balance': 0.00023610987574669955,
    'income': 8.383030443086629e-06
}

# Crear DataFrame comparativo
df_comparacion_default = pd.DataFrame({
    'Coeficiente original': coef_originales,
    'Error estándar original': errores_estandar_originales,
    'Media bootstrap': media_bootstrap,
    'Desviación estándar bootstrap': std_bootstrap
})

df_comparacion_default


,Coeficiente original,Error estándar original,Media bootstrap,Desviación estándar bootstrap
Intercept,-10.869045,0.492273,-10.931311,0.509349
C(student)[T.1],-0.646776,0.236257,-0.637073,0.245118
balance,0.005737,0.000232,0.005762,0.000236
income,0.000003,0.000008,0.000003,0.000008


**¿Por qué podría haber diferencias en los resultados?**
>
>Las diferencias aparecen porque el método clásico calcula los errores estándar bajo supuestos ideales que casi nunca se cumplen en datos reales, mientras que el bootstrap refleja la variabilidad verdadera del modelo al remuestrear directamente las observaciones. Si hay ruido, outliers, poca muestra o el modelo no encaja perfectamente con los supuestos, el bootstrap lo captura y el método tradicional no, por lo que sus resultados pueden variar.

**Agrega regularización L2 a los modelos del dataset de Advertising (optimiza el hiperparámetro). Utiliza ese valor del hiperparámetro para repetir el experimento de los 1000 remuestreos. Calcula la desviación estándar de los coeficientes obtenidos.**

In [140]:
#  Leer datos
df = pd.read_csv("Advertising.csv", index_col="Unnamed: 0")
X = df[["TV", "radio", "newspaper"]].values
y = df["sales"].values

# Definir modelo Ridge
ridge = Ridge(fit_intercept=True)

# Definir rango de búsqueda para alpha (regularización L2)
param_space = {'alpha': (1e-6, 100.0, 'log-uniform')}

# Configurar búsqueda bayesiana
bayes_search = BayesSearchCV(
    ridge,
    param_space,
    n_iter=30,  # número de iteraciones
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

# Ajustar búsqueda bayesiana
bayes_search.fit(X, y)
alpha_optimo = bayes_search.best_params_['alpha']
print("Valor óptimo de alpha (L2):", alpha_optimo)

#  Ajustar Ridge con alpha óptimo
modelo_ridge = Ridge(alpha=alpha_optimo, fit_intercept=True)
modelo_ridge.fit(X, y)

# Coeficientes
coeficientes_ridge = dict(zip(["Intercept"] + list(df[["TV", "radio", "newspaper"]].columns),
                               [modelo_ridge.intercept_] + list(modelo_ridge.coef_)))
print("Coeficientes del modelo Ridge optimizado (L2):")
print(coeficientes_ridge)


Valor óptimo de alpha (L2): 1.0691593986059968e-06
Coeficientes del modelo Ridge optimizado (L2):
{'Intercept': 2.9388893695429115, 'TV': 0.04576464545539562, 'radio': 0.18853001691294385, 'newspaper': -0.0010374930411941857}


In [131]:
# Leer datos
df = pd.read_csv("Advertising.csv", index_col="Unnamed: 0")
X = df[["TV", "radio", "newspaper"]].values
y = df["sales"].values

# Optimización bayesiana para alpha (regularización L2)
ridge = Ridge(fit_intercept=True)
param_space = {'alpha': (1e-6, 100.0, 'log-uniform')}  # rango de alpha

bayes_search = BayesSearchCV(
    ridge,
    param_space,
    n_iter=30,                # número de iteraciones
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

bayes_search.fit(X, y)
alpha_optimo = bayes_search.best_params_['alpha']
print("Valor óptimo de alpha (L2):", alpha_optimo)

# Bootstrap con Ridge usando alpha óptimo
n_boot = 1000
coef_bootstrap = []

for i in range(n_boot):
    # Remuestreo con reemplazo
    df_sample = resample(df, replace=True, n_samples=len(df), random_state=i)
    X_sample = df_sample[["TV", "radio", "newspaper"]].values
    y_sample = df_sample["sales"].values

    # Ajustar Ridge con alpha óptimo
    modelo_boot = Ridge(alpha=alpha_optimo, fit_intercept=True)
    modelo_boot.fit(X_sample, y_sample)
    
    # Guardar coeficientes (intercepto primero)
    coef_bootstrap.append(np.hstack([modelo_boot.intercept_, modelo_boot.coef_]))

# Convertir a array numpy
coef_bootstrap = np.array(coef_bootstrap)

# Calcular media y desviación estándar
media_coef = np.mean(coef_bootstrap, axis=0)
std_coef = np.std(coef_bootstrap, axis=0, ddof=1)

# Nombres de coeficientes
nombres_coef = ['Intercept', 'TV', 'radio', 'newspaper']
media_coef_dict = dict(zip(nombres_coef, media_coef))
std_coef_dict = dict(zip(nombres_coef, std_coef))

print("\nMedia de los coeficientes (bootstrap con Ridge):")
print(media_coef_dict)

print("\nDesviación estándar de los coeficientes (bootstrap con Ridge):")
print(std_coef_dict)


Valor óptimo de alpha (L2): 1.0691593986059968e-06

Media de los coeficientes (bootstrap con Ridge):
{'Intercept': 2.9529181123566977, 'TV': 0.045659706898329565, 'radio': 0.1881472151714158, 'newspaper': -0.0005439756662169745}

Desviación estándar de los coeficientes (bootstrap con Ridge):
{'Intercept': 0.3380390675195036, 'TV': 0.0019077417970230087, 'radio': 0.010955006808859653, 'newspaper': 0.00629497113298615}


In [137]:
# Datos originales OLS
coef_ols = {'const': 2.938889, 'TV': 0.045765, 'radio': 0.188530, 'newspaper': -0.001037}
stderr_ols = {'const': 0.311908, 'TV': 0.001395, 'radio': 0.008611, 'newspaper': 0.005871}
media_boot_ols = {'const': 2.952918, 'TV': 0.045660, 'radio': 0.188147, 'newspaper': -0.000544}
std_boot_ols = {'const': 0.338039, 'TV': 0.001908, 'radio': 0.010955, 'newspaper': 0.006295}

# Resultados Ridge
alpha_opt = 1.0691593986059968e-06
media_boot_ridge = {
    'Intercept': 2.9529181123566977,
    'TV': 0.045659706898329565,
    'radio': 0.1881472151714158,
    'newspaper': -0.0005439756662169745
}
std_boot_ridge = {
    'Intercept': 0.3380390675195036,
    'TV': 0.0019077417970230087,
    'radio': 0.010955006808859653,
    'newspaper': 0.00629497113298615
}

# Crear DataFrame
df_comparativo = pd.DataFrame({
    'Coeficiente original': coef_ols,
    'Error estándar original': stderr_ols,
    'Media bootstrap OLS': media_boot_ols,
    'Desviación estándar bootstrap OLS': std_boot_ols,
    'Media bootstrap Ridge': [media_boot_ridge['Intercept'],
                              media_boot_ridge['TV'],
                              media_boot_ridge['radio'],
                              media_boot_ridge['newspaper']],
    'Desviación estándar bootstrap Ridge': [std_boot_ridge['Intercept'],
                                            std_boot_ridge['TV'],
                                            std_boot_ridge['radio'],
                                            std_boot_ridge['newspaper']]
}, index=['const', 'TV', 'radio', 'newspaper'])

print("Valor óptimo de alpha (L2):", alpha_opt)
df_comparativo


Valor óptimo de alpha (L2): 1.0691593986059968e-06


,Coeficiente original,Error estándar original,Media bootstrap OLS,Desviación estándar bootstrap OLS,Media bootstrap Ridge,Desviación estándar bootstrap Ridge
const,2.938889,0.311908,2.952918,0.338039,2.952918,0.338039
TV,0.045765,0.001395,0.045660,0.001908,0.045660,0.001908
radio,0.188530,0.008611,0.188147,0.010955,0.188147,0.010955
newspaper,-0.001037,0.005871,-0.000544,0.006295,-0.000544,0.006295
